# Auto-MuJoCo-Compiler
**Upload any 3D object → automatic physics simulation**

**Run order:** Runtime → Run all. Enter your Gemini API key in Step 2 when prompted.

> After Step 1 installs packages, the kernel restarts automatically.
> Colab will continue from Step 2 on its own — just wait a few seconds.


In [ ]:
# STEP 1: Install packages
# Kernel restarts automatically after this — Colab continues from Step 2.
import subprocess, sys

pkgs = ['trimesh==4.5.3','coacd==1.0.7','mujoco==3.2.7',
        'google-genai','pygltflib','Pillow','numpy','scipy','matplotlib']

for pkg in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

print('✅ Packages installed — restarting kernel...')
from IPython import get_ipython
get_ipython().kernel.do_shutdown(restart=True)


In [ ]:
# STEP 2: Imports + Gemini API key
import os, json, zipfile
import numpy as np

os.environ['MUJOCO_GL'] = 'egl'
import subprocess
subprocess.run(['apt-get','install','-qq','-y',
                'libegl1-mesa-dev','libgles2-mesa-dev'], capture_output=True)

import trimesh, coacd, mujoco, re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image
from pathlib import Path
from io import BytesIO
from importlib.metadata import version as pkg_version
from IPython.display import display, Image as IPImage
from google import genai
from google.genai import types

# ── Enter your Gemini API key (free at aistudio.google.com) ──────────────
import getpass
GEMINI_API_KEY = getpass.getpass('🔑 Enter your Gemini API key: ')
client = genai.Client(api_key=GEMINI_API_KEY)

print(f'trimesh : {trimesh.__version__}')
print(f'coacd   : {pkg_version("coacd")}')
print(f'mujoco  : {mujoco.__version__}')
print(f'genai   : {pkg_version("google-genai")}')

OUT = Path('/content/output')
OUT.mkdir(exist_ok=True)
print('✅ Ready')


In [ ]:
# STEP 3: Upload your .glb or .obj file
from google.colab import files
print('📂 Upload your .glb or .obj file:')
uploaded   = files.upload()
filename   = list(uploaded.keys())[0]
INPUT_PATH = Path(f'/content/{filename}')
with open(INPUT_PATH,'wb') as f: f.write(uploaded[filename])
print(f'✅ {INPUT_PATH}  ({INPUT_PATH.stat().st_size/1024:.1f} KB)')


In [ ]:
# STEP 4: Load mesh
def load_mesh(path):
    loaded = trimesh.load(str(path), force='mesh')
    if isinstance(loaded, trimesh.Scene):
        meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh)]
        mesh   = trimesh.util.concatenate(meshes)
    else:
        mesh = loaded
    mesh.process(validate=True)
    mesh.fill_holes()
    print(f'  Vertices   : {len(mesh.vertices):,}')
    print(f'  Faces      : {len(mesh.faces):,}')
    print(f'  Watertight : {mesh.is_watertight}')
    return mesh

mesh = load_mesh(INPUT_PATH)
print('✅ Mesh loaded')


In [ ]:
# STEP 5: Render 4 views with matplotlib
def render_views(mesh, n=4):
    v = mesh.vertices.copy()
    v -= v.mean(axis=0)
    v /= np.abs(v).max()
    images = []
    for az in np.linspace(0, 360, n, endpoint=False):
        fig = plt.figure(figsize=(5,5), facecolor='#222222')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#222222')
        ax.add_collection3d(Poly3DCollection(v[mesh.faces], alpha=0.92,
                            linewidths=0, facecolors='#c8a87a', edgecolors='none'))
        ax.set_xlim(-1.1,1.1); ax.set_ylim(-1.1,1.1); ax.set_zlim(-1.1,1.1)
        ax.set_axis_off(); ax.view_init(elev=25, azim=az)
        buf = BytesIO()
        plt.savefig(buf,format='png',dpi=100,bbox_inches='tight',facecolor='#222222')
        plt.close(fig); buf.seek(0)
        images.append(Image.open(buf).convert('RGB').copy())
    return images

views = render_views(mesh)
for i,v in enumerate(views): v.save(OUT/f'view_{i}.png')
row = Image.new('RGB',(500*4,500))
for i,img in enumerate(views): row.paste(img.resize((500,500)),(500*i,0))
display(row.resize((1200,300)))
print('✅ 4 views rendered')


In [ ]:
# STEP 6: Gemini 2.5 Flash — infer physics properties
PROMPT = """You are a physics property estimator for a simulation engine.
Given 4 rendered views of a 3D object, return ONLY valid JSON — no markdown, no backticks.
Start with { and end with }

{
  \"object_name\": \"<short name>\",
  \"material\": \"<ceramic|wood|metal|plastic|rubber|glass|fabric|stone>\",
  \"mass_kg\": <float>,
  \"friction\": <float 0.0-1.0>,
  \"restitution\": <float 0.0-1.0>,
  \"reasoning\": \"<one sentence>\"
}"""

def pil_to_part(img):
    buf=BytesIO(); img.save(buf,format='PNG')
    return types.Part.from_bytes(data=buf.getvalue(),mime_type='image/png')

contents=[types.Part.from_text(text=PROMPT)]+[pil_to_part(v) for v in views]
resp=client.models.generate_content(
    model='gemini-2.5-flash',contents=contents,
    config=types.GenerateContentConfig(temperature=0.0,max_output_tokens=2048)
)
raw=resp.text.strip()
print('── Raw Gemini response ──────────────────────────')
print(raw)
print('────────────────────────────────────────────────')
match=re.search(r'\{.*\}',raw,re.DOTALL)
if not match: raise ValueError(f'No JSON in response:\n{raw}')
props=json.loads(match.group())
with open(OUT/'physics_properties.json','w') as f: json.dump(props,f,indent=2)
print('\n── Physics Tags ──────────────────────────────────')
for k,val in props.items(): print(f'  {k:15s}: {val}')
print('✅ VLM tagging done')


In [ ]:
# STEP 7: CoACD convex decomposition
cm    = coacd.Mesh(mesh.vertices, mesh.faces)
parts = coacd.run_coacd(cm,threshold=0.05,max_convex_hull=32,preprocess_mode='auto')
hulls = []
for i,(v,f) in enumerate(parts):
    h=trimesh.Trimesh(vertices=np.array(v),faces=np.array(f))
    p=OUT/f'hull_{i:02d}.obj'; h.export(str(p))
    hulls.append((h,p))
    print(f'  hull_{i:02d}: {len(h.vertices)} verts  vol={h.volume:.6f}')
print(f'✅ {len(hulls)} hull(s) generated')


In [ ]:
# STEP 8: Build MuJoCo scene
assets={}
vis_path=OUT/'visual.obj'; mesh.export(str(vis_path))
assets['visual.obj']=vis_path.read_bytes()
hull_names=[]
for i,(h,hp) in enumerate(hulls):
    nm=f'hull_{i:02d}'; assets[f'{nm}.obj']=hp.read_bytes(); hull_names.append(nm)

spawn_z=float(0.5-mesh.bounds[0][2])
mass=float(props['mass_kg']); fric=float(props['friction'])
total_vol=sum(h.volume for h,_ in hulls) or 1.0

mesh_assets='<mesh name="vis_mesh" file="visual.obj"/>\n    '
mesh_assets+='\n    '.join(f'<mesh name="{nm}" file="{nm}.obj"/>' for nm in hull_names)

col_geoms='\n      '.join(
    f'<geom name="col_{i:02d}" type="mesh" mesh="{nm}" '
    f'mass="{mass*(hulls[i][0].volume/total_vol):.6f}" '
    f'friction="{fric} 0.005 0.0001" margin="0.001" rgba="0 0 0 0"/>'
    for i,nm in enumerate(hull_names)
)

xml=f"""<mujoco model="{props['object_name'].replace(' ','_')}">
  <compiler angle="radian"/>
  <option gravity="0 0 -9.81" timestep="0.002" integrator="implicitfast"
          cone="elliptic" noslip_iterations="5"/>
  <asset>
    {mesh_assets}
    <texture name="grid" type="2d" builtin="checker" width="512" height="512"
             rgb1="0.85 0.85 0.85" rgb2="0.65 0.65 0.65"/>
    <material name="floor_mat" texture="grid" texrepeat="4 4"/>
  </asset>
  <worldbody>
    <light name="top" pos="0 0 5" dir="0 0 -1" castshadow="true"/>
    <geom name="floor" type="plane" size="8 8 0.1"
          material="floor_mat" friction="{fric} 0.005 0.0001"/>
    <body name="object" pos="0 0 {spawn_z:.6f}">
      <freejoint name="free"/>
      <geom name="visual" type="mesh" mesh="vis_mesh"
            contype="0" conaffinity="0" rgba="0.78 0.70 0.55 1"/>
      {col_geoms}
    </body>
  </worldbody>
</mujoco>"""

with open(OUT/'scene.xml','w') as f: f.write(xml)
model=mujoco.MjModel.from_xml_string(xml,assets)
data=mujoco.MjData(model)
print(f'✅ Compiled — {model.nbody} bodies | {model.ngeom} geoms | {model.nv} DoF')
print(f'   mass={mass}kg  friction={fric}  spawn_z={spawn_z:.4f}')


In [ ]:
# STEP 9: Simulate 3s
renderer=mujoco.Renderer(model,height=480,width=640)
cam=mujoco.MjvCamera()
cam.type=mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat=[0,0,0.3]; cam.distance=2.0; cam.elevation=-25; cam.azimuth=45
opt=mujoco.MjvOption(); mujoco.mjv_defaultOption(opt)
skip=max(1,int(1.0/(30*model.opt.timestep)))
steps=int(3.0/model.opt.timestep); frames=[]
print(f'Simulating 3s ({steps:,} steps)...')
for s in range(steps):
    mujoco.mj_step(model,data)
    if s%skip==0:
        renderer.update_scene(data,camera=cam,scene_option=opt)
        frames.append(renderer.render().copy())
    if s%10000==0:
        print(f'  t={data.time:.2f}s | z={data.qpos[2]:.4f}m | contacts={data.ncon}')
renderer.close()
print(f'✅ {len(frames)} frames')


In [ ]:
# STEP 10: Save GIF
gif=OUT/'simulation.gif'
pf=[Image.fromarray(f) for f in frames]
pf[0].save(str(gif),save_all=True,append_images=pf[1:],duration=int(1000/30),loop=0)
display(IPImage(filename=str(gif)))
print('\n── Physics Summary ────────────────────────────')
for k,val in props.items(): print(f'  {k:15s}: {val}')
print('\n✅ Done! Run Step 11 to download everything.')


In [ ]:
# STEP 11: Download output zip
# Extract locally, put viewer.py in same folder, run: python viewer.py
from google.colab import files as colab_files
zip_path='/content/output.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,arcname=f.relative_to(OUT))
print('📦 Contents:')
with zipfile.ZipFile(zip_path) as z:
    for n in sorted(z.namelist()):
        print(f'  {n:35s} {z.getinfo(n).file_size/1024:6.1f} KB')
colab_files.download(zip_path)
print('\n✅ Download started')
print('\nLocal viewer:')
print('  pip install mujoco==3.2.7 trimesh numpy')
print('  python viewer.py')
